# Tutorial 01 — AntibodyTokenizer

This notebook walks through the 25-token amino-acid vocabulary used throughout AbLLaVA.

**Vocab layout**

| ID | Token |
|----|-------|
| 0  | [PAD] |
| 1  | [BOS] |
| 2  | [EOS] |
| 3  | [SEP] |
| 4  | [MASK] |
| 5–24 | ACDEFGHIKLMNPQRSTVWY |

No external dependencies required — just PyTorch.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.utils.tokenizer import AntibodyTokenizer, AA_VOCAB, SPECIAL_TOKENS
import torch

## 1. Inspect the vocabulary

In [ ]:
tok = AntibodyTokenizer()
print(f"Vocab size : {len(tok)}")
print(f"Special IDs: PAD={tok.pad_id}, BOS={tok.bos_id}, EOS={tok.eos_id}, SEP={tok.sep_id}, MASK={tok.mask_id}")
print(f"AA tokens  : {AA_VOCAB}")
print()
print("Full token→id mapping:")
for token, id_ in sorted(tok.token_to_id.items(), key=lambda x: x[1]):
    print(f"  {id_:2d}  {token}")

## 2. Encode a single sequence

In [ ]:
heavy = "EVQLVESGGGLVQPGGSLRLSCAASGFTFS"
ids = tok.encode(heavy)
print(f"Sequence : {heavy}")
print(f"Encoded  : {ids}")
print(f"Decoded  : {tok.decode(ids)}")
assert tok.decode(ids) == heavy, "Round-trip failed!"
print("Round-trip OK")

## 3. Encode a heavy+light pair with special tokens

In [ ]:
light = "DIQMTQSPSSLSASVGDRVTITC"
pair_ids = tok.encode_pair(heavy, light)
print(f"pair_ids length: {len(pair_ids)}")
print(f"pair_ids: {pair_ids[:8]} ... {pair_ids[-4:]}")

# Verify structure: [BOS] heavy [SEP] light [EOS]
assert pair_ids[0] == tok.bos_id
assert pair_ids[-1] == tok.eos_id
sep_pos = pair_ids.index(tok.sep_id)
print(f"[SEP] at position {sep_pos}  (heavy length = {len(heavy)})")

## 4. Split a pair back to heavy / light

In [ ]:
h_out, l_out = tok.split_pair_ids(pair_ids)
print(f"Heavy : {h_out}")
print(f"Light : {l_out}")
assert h_out == heavy
assert l_out == light
print("Split round-trip OK")

## 5. Batch padding

In [ ]:
seqs = [
    tok.encode_pair("EVQLVES", "DIQMTQ"),
    tok.encode_pair("EVQLVESGGGLVQ", "DIQMTQSPSS"),
    tok.encode_pair("EVQLVESGGGLVQPGGSL", "DI"),
]
input_ids, attn_mask = tok.pad_batch(seqs)
print(f"input_ids shape  : {input_ids.shape}")
print(f"attn_mask shape  : {attn_mask.shape}")
print(f"Padding token id : {tok.pad_id}")
print()
print("input_ids:")
print(input_ids)
print()
print("attention_mask (1=real, 0=pad):")
print(attn_mask)

## 6. MASK token demo (for CDR-infill pre-training)

In [ ]:
ids = tok.encode_pair(heavy, light)
# Mask positions 5-10 (within heavy chain)
masked = ids.copy()
for pos in range(5, 11):
    masked[pos] = tok.mask_id

print(f"Original : {tok.decode(ids, skip_special=False)[:50]}")
print(f"Masked   : {tok.decode(masked, skip_special=False)[:50]}")

---
**Summary**: `AntibodyTokenizer` handles encode/decode, paired sequences, batch padding, and masking — all the token-level operations needed by the decoder.